# OpenVLA drives LIBERO on Colab, with a recorded video

Everything heavy runs in a **Python 3.10 venv** at `/content/venv` (Colab's own Python 3.13 can't install LIBERO's pins).
Notebook cells only install things and launch scripts with `/content/venv/bin/python`; the final cell plays the recorded video.

Use a GPU runtime, then Run all. Needs ~25 GB free disk. The model is loaded in 4-bit (~5 GB) so it fits a 15 GB T4.

**If your session still has `/content/venv`, the LIBERO clone and the weights, you only need the last three cells**
(write script, run script, play video).

## 1. GPU and disk

In [1]:
!nvidia-smi
!df -h /content | tail -1

Tue Oct 13 12:45:24 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Python 3.10 environment

In [2]:
import os
!pip install -q uv
!uv venv /content/venv --python 3.10
!apt-get install -y -qq libgl1 libegl1 libosmesa6 patchelf

os.environ["UV_PYTHON"] = "/content/venv/bin/python"   # every `!uv pip install` targets the venv
PY = "/content/venv/bin/python"
!{PY} --version

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.9/17.9 MB 103.6 MB/s eta 0:00:0000:010:01
Using CPython 3.10.22
Creating virtual environment at: venv
Activate with: source venv/bin/activate
Selecting previously unselected package patchelf.
(Reading database ... 127332 files and directories currently installed.)
Preparing to unpack .../patchelf_0.18.0-1.1build1_amd64.deb ...
Unpacking patchelf (0.18.0-1.1build1) ...
Selecting previously unselected package libosmesa6:amd64.
Preparing to unpack .../libosmesa6_25.1.7-1ubuntu2~24.04.2_amd64.deb ...
Unpacking libosmesa6:amd64 (25.1.7-1ubuntu2~24.04.2) ...
Setting up patchelf (0.18.0-1.1build1) ...
Setting up libosmesa6:amd64 (25.1.7-1ubuntu2~24.04.2) ...
Processing triggers for man-db (2.12.0-4build2) ...
Processing triggers for libc-bin (2.39-0ubuntu8.9) ...
/sbin/ldconfig.real: /usr/local/lib/libtbbmalloc_proxy.so.2 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbb.so.12 is not a symbolic link

/sbin/ldconfig.real: /usr/loc

## 3. Install packages into the venv

In [3]:
!uv pip install torch torchvision

Using Python 3.10.22 environment at: venv
Resolved 32 packages in 319ms                                        
Prepared 32 packages in 53.51s                                           
Installed 32 packages in 347ms                              
 + cuda-bindings==13.4.3
 + cuda-pathfinder==1.8.3
 + cuda-toolkit==13.0.3.0
 + filelock==4.1.0
 + fsspec==2026.9.0
 + jinja2==3.1.6
 + markupsafe==3.0.4
 + mpmath==1.3.0
 + networkx==3.4.2
 + numpy==2.2.6
 + nvidia-cublas==13.1.1.3
 + nvidia-cuda-cupti==13.0.85
 + nvidia-cuda-nvrtc==13.0.88
 + nvidia-cuda-runtime==13.0.96
 + nvidia-cudnn-cu13==9.24.0.43
 + nvidia-cufft==12.0.0.61
 + nvidia-cufile==1.15.1.6
 + nvidia-curand==10.4.0.35
 + nvidia-cusolver==12.0.4.66
 + nvidia-cusparse==12.6.3.3
 + nvidia-cusparselt-cu13==0.8.1
 + nvidia-nccl-cu13==2.30.7
 + nvidia-nvjitlink==13.4.92
 + nvidia-nvshmem-cu13==3.4.5
 + nvidia-nvtx==13.0.85
 + pillow==12.3.0
 + setuptools==84.0.0
 + sympy==1.14.0
 + torch==2.14.1
 + torchvision==0.29.1
 + triton==3.8

In [4]:
# LIBERO requirements, minus lines that clash with the model stack (torch*, transformers, einops).
!git clone https://github.com/Lifelong-Robot-Learning/LIBERO /content/LIBERO
!grep -viE "^(torch|torchvision|torchaudio|transformers|einops)" /content/LIBERO/requirements.txt > /content/libero_reqs.txt
!uv pip install -r /content/libero_reqs.txt

Cloning into '/content/LIBERO'...
remote: Enumerating objects: 1788, done.
remote: Total 1788 (delta 0), reused 0 (delta 0), pack-reused 1788 (from 1)
Receiving objects: 100% (1788/1788), 316.00 MiB | 31.12 MiB/s, done.
Resolving deltas: 100% (746/746), done.
Updating files: 100% (1116/1116), done.
Using Python 3.10.22 environment at: venv
Resolved 121 packages in 6.05s                                       
Prepared 90 packages in 18.33s                                           
Uninstalled 1 package in 21ms
Installed 90 packages in 330ms                              
 + absl-py==2.5.1
 + antlr4-python3-runtime==4.9.3
 + attrs==26.1.0
 + bddl==1.0.1
 + certifi==2026.7.22
 + charset-normalizer==3.5.2
 + click==8.5.0
 + cloudpickle==2.1.0
 + cycler==0.12.1
 + docker-pycreds==0.4.0
 + easydict==1.9
 + egl-probe==1.0.2
 + einops==0.4.1
 + etils==1.13.0
 + exceptiongroup==1.3.1
 + fastjsonschema==2.22.2
 + fonttools==4.65.0
 + future==0.18.2
 + gitdb==4.0.12
 + gitpython==3.2.0
 + glfw==2

In [5]:
# OpenVLA stack + MuJoCo pin (robosuite 1.4.0 breaks on MuJoCo 3.x) + 4-bit loading + video writing.
!uv pip install "transformers==4.40.1" "tokenizers==0.19.1" "timm==0.9.10" accelerate sentencepiece einops huggingface_hub
!uv pip install "mujoco==2.3.7" bitsandbytes imageio imageio-ffmpeg
!uv pip install --no-deps -e /content/LIBERO

Using Python 3.10.22 environment at: venv
Resolved 51 packages in 220ms                                        
Prepared 6 packages in 518ms                                             
Uninstalled 2 packages in 17ms
Installed 6 packages in 53ms                                
 + accelerate==1.15.0
 + safetensors==0.8.0
 + sentencepiece==0.2.2
 + timm==0.9.10
 - tokenizers==0.12.1
 + tokenizers==0.19.1
 - transformers==4.21.1
 + transformers==4.40.1
Using Python 3.10.22 environment at: venv
Resolved 39 packages in 106ms                                        
Prepared 2 packages in 832ms                                             
Uninstalled 1 package in 7ms
Installed 2 packages in 3ms                                 
 + bitsandbytes==0.50.2
 - mujoco==3.15.0
 + mujoco==2.3.7
Using Python 3.10.22 environment at: venv
Resolved 1 package in 598ms                                          
Prepared 1 package in 253ms                                              
Installed 1 package in 3m

## 4. Verify the environment

In [6]:
%%writefile /content/check_env.py
import torch, transformers, tokenizers, timm, numpy, mujoco, bitsandbytes, imageio
print("torch", torch.__version__, "| transformers", transformers.__version__,
      "| tokenizers", tokenizers.__version__, "| timm", timm.__version__,
      "| numpy", numpy.__version__, "| mujoco", mujoco.__version__,
      "| bitsandbytes", bitsandbytes.__version__)
print("cuda available:", torch.cuda.is_available(), "| torch cuda:", torch.version.cuda)
from transformers import AutoProcessor, AutoModelForVision2Seq  # must import cleanly
x = torch.randn(1024, 1024, device="cuda")
print("matmul ok:", (x @ x).sum().item())

Writing /content/check_env.py


In [7]:
!{PY} /content/check_env.py

/content/venv/lib/python3.10/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
/content/venv/lib/python3.10/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
/content/venv/lib/python3.10/site-packages/torch/jit/_script.py:1643: FutureWarning: `torch.jit.interface` is deprecated. Please use `torch.compile` instead.
  warnings.warn(
torch 2.14.1+cu130 | transformers 4.40.1 | tokenizers 0.19.1 | timm 0.9.10 | numpy 1.22.4 | mujoco 2.3.7 | bitsandbytes 0.50.2
cuda available: True | torch cuda: 13.0
matmul ok: -22001.23828125


## 5. Download the fine-tuned OpenVLA checkpoint (~14 GB)

In [8]:
%%writefile /content/download_weights.py
from huggingface_hub import snapshot_download
d = snapshot_download(repo_id="openvla/openvla-7b-finetuned-libero-object",
                      local_dir="/content/weights/openvla-7b-finetuned-libero-object")
print("Weights at:", d)

Writing /content/download_weights.py


In [9]:
!{PY} /content/download_weights.py
!ls -la /content/weights/openvla-7b-finetuned-libero-object


















Weights at: /content/weights/openvla-7b-finetuned-libero-object
total 14731576
drwxr-xr-x 3 root root       4096 Oct 13 12:50 .
drwxr-xr-x 3 root root       4096 Oct 13 12:50 ..
-rw-r--r-- 1 root root         21 Oct 13 12:47 added_tokens.json
drwxr-xr-x 3 root root       4096 Oct 13 12:47 .cache
-rw-r--r-- 1 root root       3552 Oct 13 12:47 config.json
-rw-r--r-- 1 root root       2153 Oct 13 12:47 dataset_statistics.json
-rw-r--r-- 1 root root        136 Oct 13 12:47 generation_config.json
-rw-r--r-- 1 root root       1519 Oct 13 12:47 .gitattributes
-rw-r--r-- 1 root root 4925122448 Oct 13 12:49 model-00001-of-00004.safetensors
-rw-r--r-- 1 root root 4947392496 Oct 13 12:50 model-00002-of-00004.safetensors
-rw-r--r-- 1 root root 4947417456 Oct 13 12:50 model-00003-of-00004.safetensors
-rw-r--r-- 1 root root  262668432 Oct 13 12:47 model-00004-of-00004.safetensors
-rw-r--r-- 1 root root      94764 Oct 13 12:47 model.safetensors.index.json
-rw-r--r-- 1 root root      

## 6. LIBERO config

In [10]:
import os
os.makedirs(os.path.expanduser("~/.libero"), exist_ok=True)
root = "/content/LIBERO/libero/libero"
cfg = f"""benchmark_root: {root}
bddl_files: {root}/bddl_files
init_states: {root}/init_files
datasets: /content/LIBERO/libero/datasets
assets: {root}/assets
"""
open(os.path.expanduser("~/.libero/config.yaml"), "w").write(cfg)
print(cfg)

benchmark_root: /content/LIBERO/libero/libero
bddl_files: /content/LIBERO/libero/libero/bddl_files
init_states: /content/LIBERO/libero/libero/init_files
datasets: /content/LIBERO/libero/datasets
assets: /content/LIBERO/libero/libero/assets



## 7. Run OpenVLA in the simulator and record a video

Records the agent camera and the wrist camera side by side to `/content/rollout.mp4`, even if the run crashes partway.
Flags to flip if behaviour looks wrong: `INVERT_GRIPPER`, `ROTATE_180`. Set `TASK_ID` (0-9) to try other `libero_object` tasks.

In [11]:
%%writefile /content/run_openvla_libero.py
import os, sys, json
os.environ["MUJOCO_GL"] = "egl"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
sys.path.insert(0, "/content/LIBERO")
import functools
import numpy as np, torch, imageio
from PIL import Image
import transformers.modeling_utils as _mu
from transformers import AutoProcessor, AutoModelForVision2Seq, BitsAndBytesConfig
from libero.libero import benchmark, get_libero_path
from libero.libero.envs import OffScreenRenderEnv

MODEL_DIR = "/content/weights/openvla-7b-finetuned-libero-object"
QUANT = "4bit"           # "4bit" (~5 GB), "8bit" (~9 GB), or "bf16" (needs a 24 GB GPU)
TASK_ID = 2              # which libero_object task (0-9)
INIT_STATE_ID = 0        # which fixed starting state for that task
MAX_STEPS = 300
INVERT_GRIPPER = True    # flip to False if the gripper behaves backwards
ROTATE_180 = True        # flip to False to only flip vertically
VIDEO_PATH = "/content/rollout.mp4"

stats = json.load(open(f"{MODEL_DIR}/config.json")).get("norm_stats", {})
UNNORM_KEY = next((k for k in stats if k.startswith("libero_object")), None)
assert UNNORM_KEY, f"no libero_object key; found {list(stats.keys())}"
print("using unnorm_key:", UNNORM_KEY, "| quantization:", QUANT)

# Workaround: new accelerate calls model.to(device) after loading, which old transformers (4.40.1)
# forbids for bitsandbytes models. With a single GPU the weights are already placed, so skip it.
_orig_dispatch = _mu.dispatch_model
def _dispatch_skip_single_device(model, **kw):
    dm = kw.get("device_map")
    if isinstance(dm, dict) and len(set(dm.values())) == 1:
        return model
    return _orig_dispatch(model, **kw)
_mu.dispatch_model = _dispatch_skip_single_device

processor = AutoProcessor.from_pretrained(MODEL_DIR, trust_remote_code=True)
kwargs = dict(torch_dtype=torch.bfloat16, low_cpu_mem_usage=True, trust_remote_code=True)
if QUANT == "4bit":
    kwargs["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16)
    kwargs["device_map"] = {"": 0}
elif QUANT == "8bit":
    kwargs["quantization_config"] = BitsAndBytesConfig(load_in_8bit=True)
    kwargs["device_map"] = {"": 0}
model = AutoModelForVision2Seq.from_pretrained(MODEL_DIR, **kwargs)
if QUANT == "bf16":
    model = model.to("cuda")
model.eval()

# The skipped .to() also moved small non-quantized tensors (e.g. rotary inv_freq) to the GPU.
moved_b = moved_p = 0
for mod in model.modules():
    for k, b in list(mod._buffers.items()):
        if b is not None and b.device.type != "cuda":
            mod._buffers[k] = b.to("cuda"); moved_b += 1
    for k, p in list(mod._parameters.items()):
        if p is not None and p.device.type != "cuda":
            p.data = p.data.to("cuda"); moved_p += 1
print(f"moved {moved_b} buffers and {moved_p} params from cpu to cuda", flush=True)
print(f"GPU memory after load: {torch.cuda.memory_allocated()/1e9:.1f} GB", flush=True)

suite = benchmark.get_benchmark_dict()["libero_object"]()
task = suite.get_task(TASK_ID)
bddl = os.path.join(get_libero_path("bddl_files"), task.problem_folder, task.bddl_file)
env = OffScreenRenderEnv(bddl_file_name=bddl, camera_heights=256, camera_widths=256)
env.seed(0)
env.reset()

# LIBERO's init-state file is a pickle with numpy arrays; torch>=2.6 refuses that by default.
# It ships in the LIBERO repo we cloned, so allow it for this one call only.
_orig_load = torch.load
torch.load = functools.partial(_orig_load, weights_only=False)
try:
    init_states = suite.get_task_init_states(TASK_ID)
finally:
    torch.load = _orig_load
obs = env.set_init_state(init_states[INIT_STATE_ID])

for _ in range(10):                                           # let objects settle
    obs, _, _, _ = env.step([0, 0, 0, 0, 0, 0, -1])

instruction = task.language.lower()
print("instruction:", instruction)

def prep_image(frame):
    frame = frame[::-1, ::-1] if ROTATE_180 else frame[::-1]
    return Image.fromarray(frame.copy())

def video_frame(obs):
    # agent camera (+ wrist camera if present), rotated upright, side by side
    views = [obs["agentview_image"][::-1, ::-1]]
    if "robot0_eye_in_hand_image" in obs:
        views.append(obs["robot0_eye_in_hand_image"][::-1, ::-1])
    return np.ascontiguousarray(np.concatenate(views, axis=1))

frames = [video_frame(obs)]
success = False
step = -1
try:
    for step in range(MAX_STEPS):
        image = prep_image(obs["agentview_image"])
        prompt = f"In: What action should the robot take to {instruction}?\nOut:"
        inputs = processor(prompt, image).to("cuda", dtype=torch.bfloat16)
        action = model.predict_action(**inputs, unnorm_key=UNNORM_KEY, do_sample=False)
        action = np.array(action, dtype=np.float64)

        action[-1] = 1.0 if action[-1] > 0.5 else -1.0        # binarize gripper to {-1, +1}
        if INVERT_GRIPPER:
            action[-1] *= -1.0

        obs, reward, done, info = env.step(action.tolist())
        frames.append(video_frame(obs))
        print(f"step={step:03d} action={np.round(action, 3).tolist()} reward={reward}", flush=True)
        if done:
            success = reward > 0
            break
finally:
    imageio.mimsave(VIDEO_PATH, frames, fps=20)
    print(f"saved {len(frames)} frames to {VIDEO_PATH}", flush=True)

print("SUCCESS" if success else "NOT SUCCESSFUL", "after", step + 1, "steps")
env.close()

Writing /content/run_openvla_libero.py


In [12]:
!{PY} /content/run_openvla_libero.py 2>&1 | grep -vE "Warning|warn|WARNING|Gym|gymnasium|Please upgrade|migration"

  sentry_hub = sentry_sdk.Hub(sentry_client)
A new version of the following files was downloaded from https://huggingface.co/openvla/openvla-7b:
- processing_prismatic.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
A new version of the following files was downloaded from https://huggingface.co/openvla/openvla-7b:
- configuration_prismatic.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
A new version of the following files was downloaded from https://huggingface.co/openvla/openvla-7b:
- modeling_prismatic.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
using unnorm_key: libero_object | quantization: 4bit
Loading checkpoint shards: 100%|██████████| 4/4 [01:02<00:00, 15.67s/it]


## 8. Play the recorded video (runs in the normal notebook kernel)

In [13]:
from IPython.display import Video
Video("/content/rollout.mp4", embed=True, width=800, html_attributes="controls loop autoplay muted")

If the player doesn't show, download `/content/rollout.mp4` from Colab's file browser (folder icon, left side).